# 01 BFS（广度优先搜索）：条目、证明要点、逐行实现


本课按百科条目写：先定义，再数据结构，再完整手算，再 **一行一行写出算法**。调度台在最后，可跳过。

---

## 1. 定义

**广度优先搜索**（Breadth-First Search, BFS）在图上从起点出发，按「离起点的边数」从小到大访问顶点。

- 输入：图 \(G=(V,E)\)，起点 \(s\)，终点 \(t\)。本课每条边的长度都是 **1**（无权图，或等权图）。
- 输出：一条 \(s\) 到 \(t\) 的路径，边数最少；若不可达则失败。

它 1959 年前后由 Moore 以及 Lee（电路布线）给出。路径规划里，格子图「每步代价相同」时，BFS 就是最短路算法。

**不是** 最短路程算法：边有不同长度时要用 Dijkstra。

---

## 2. 和 DFS 差在哪

| | BFS | DFS（深度优先） |
|--|--|--|
| 待处理集合 | **队列** FIFO | **栈** LIFO / 递归 |
| 先处理谁 | 最早发现的、离起点近的 | 最新发现的、往深处扎 |
| 等权最短路 | 是 | 否 |
| 典型用途 | 最短步数、分层 | 拓扑、连通、迷宫「一条路走到底」 |

只换「从哪端取点」，行为完全不同。实现时用错 `pop()` / `popleft()` 是最常见的 bug。

---

## 3. 不变量（为什么对）

维护：

- `dist[v]`：从 \(s\) 到 \(v\) 的最少边数（已知）
- `came_from[v]`：那条最短路上 \(v\) 的前驱
- 队列里的点按 `dist` **非降** 排列

**引理：** 每个点第一次被发现（写入 `came_from` 并入队）时，`dist` 已经是最优边数。

理由：边权全是 1，先处理完所有 `dist = k` 的点，才会处理 `dist = k+1`。第一次碰到 \(t\) 时，不存在更短的没处理完的层。

因此：**发现终点就可以停**，不必把整张图扫完。这也是为什么「每个点只入队一次」。

---

## 4. 伪代码（与实现一一对应）

```
function BFS(G, s, t):
    queue ← 空队列
    enqueue(queue, s)
    came_from[s] ← NIL
    dist[s] ← 0
    while queue 非空:
        u ← dequeue(queue)          # 队头，最早入队的
        if u = t:
            return Reconstruct(came_from, t)
        for each 边 (u, v) in G:
            if v 从未出现在 came_from:
                came_from[v] ← u
                dist[v] ← dist[u] + 1
                enqueue(queue, v)
    return FAILURE
```

**何时标记「见过」：** 必须在 **入队时** 标记，不能等出队再标。否则同一点会进队多次，复杂度和正确性都坏。本实现用 `came_from` 兼作 visited。

**复杂度：** 每点最多入队一次，每条边最多看一次，时间 \(O(|V|+|E|)\)。网格里 \(|E|\le 4|V|\)，即与格子数成正比。空间也是 \(O(|V|)\)。

---

## 5. 完整手算（请先算，再对答案）

地图 \(4\times 3\)。坐标 **x 向右、y 向下**。墙在 (1,1)。S=(0,0)，G=(3,2)。邻居顺序：**右、左、下、上**。

```
列x  0 1 2 3
y=0  S . . .
y=1  . # . .
y=2  . . . G
```

每一行是一次 `dequeue`：

| 出队 u | dist | 新入队的邻居 | 队列（左=队头） | came_from 新写 |
|--------|------|----------------|-----------------|----------------|
| (0,0) | 0 | (1,0), (0,1) | (1,0),(0,1) | (1,0)←(0,0), (0,1)←(0,0) |
| (1,0) | 1 | (2,0) | (0,1),(2,0) | (2,0)←(1,0) |
| (0,1) | 1 | (0,2) | (2,0),(0,2) | (0,2)←(0,1) |
| (2,0) | 2 | (3,0),(2,1) | (0,2),(3,0),(2,1) | |
| (0,2) | 2 | (1,2) | (3,0),(2,1),(1,2) | |
| (3,0) | 3 | (3,1) | (2,1),(1,2),(3,1) | |
| (2,1) | 3 | | (1,2),(3,1) | (2,1) 的右是 (3,1) 已在队 |
| (1,2) | 3 | (2,2) | (3,1),(2,2) | |
| (3,1) | 4 | (3,2)=G | (2,2),(3,2) | G←(3,1) |
| (2,2) | 4 | | (3,2) | |
| **(3,2)** | **5** | 停 | | |

回溯：`(3,2)←(3,1)←(3,0)←(2,0)←(1,0)←(0,0)`，边数 5。  
另一条 `(0,0)-(0,1)-(0,2)-(1,2)-(2,2)-(3,2)` 也是 5 条边；先发现哪条取决于邻居顺序，**边数相同**。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：地图 API

下面四段代码组成完整 BFS。每段只做一件事。先把图变成「给一个点，能列出邻居」。


In [ ]:
from collections import deque
from lib.pretty import draw_grid, print_step

# GRID[y][x]：0 可走，1 墙。不要写成 GRID[x][y]，和图像习惯相反。
GRID = [
    [0, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 0],
]
H, W = len(GRID), len(GRID[0])
START, GOAL = (0, 0), (3, 2)
OBSTACLES = {(x, y) for y in range(H) for x in range(W) if GRID[y][x] == 1}


def in_bounds(x, y):
    # 防止 x=-1 或 x=W 时 Python 用负数下标「绕到最后一列」——那是静默错路。
    return 0 <= x < W and 0 <= y < H


def passable(pos):
    x, y = pos
    return in_bounds(x, y) and GRID[y][x] == 0


def neighbors(pos):
    # 返回 [(邻点, 边权), ...]。本课边权恒 1。
    # 顺序写死：右、左、下、上。换顺序只换并列最短路的选择，不换边数。
    x, y = pos
    out = []
    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        nxt = (x + dx, y + dy)
        if passable(nxt):
            out.append((nxt, 1))
    return out


print(draw_grid(W, H, obstacles=OBSTACLES, start=START, goal=GOAL))
print("S 的邻居应是 (1,0) 和 (0,1):", neighbors(START))
print("(1,0) 不能往下进墙:", neighbors((1, 0)))


`in_bounds` 必须在读 `GRID[y][x]` **之前**调用。只写 `GRID[y][x]==0` 会在越界时抛异常或（若用 `arr[-1]`）走到错误格子。

`neighbors` 不返回墙、不返回界外。BFS 本体不再判断「能不能走」，只判断「见过没有」。职责分离：图 API vs 搜索。

---

## 7. 实现：回溯

不把整条路径跟着每个点复制。每个点只记一个父亲。终点找到后往回爬。

`came_from[s] is None` 是爬升终止条件。若忘记给起点赋值 `None`，回溯会 `KeyError` 或死循环。


In [ ]:
def reconstruct(came_from, goal):
    # 从 goal 沿 came_from 走到 start。因为是往回走，最后 reverse。
    node = goal
    path = [node]
    while came_from[node] is not None:
        node = came_from[node]
        path.append(node)
    path.reverse()
    return path


fake = {
    (0, 0): None,
    (1, 0): (0, 0),
    (2, 0): (1, 0),
    (3, 0): (2, 0),
    (3, 1): (3, 0),
    (3, 2): (3, 1),
}
print("应打印手算那条路径:", reconstruct(fake, (3, 2)))


## 8. 实现：BFS 主循环（逐行）

| 行 | 代码在干什么 |
|----|----------------|
| `queue = deque([start])` | 双端队列。只在右边 `append`、左边 `popleft` ⇒ FIFO。 |
| `came_from = {start: None}` | 起点已发现；值 None 表示回溯到此结束。兼作 visited。 |
| `dist[start] = 0` | 起点边数 0。 |
| `while queue` | 还有没扩展的点。空了还没碰到 t ⇒ 不可达。 |
| `current = queue.popleft()` | **扩展**：第一次从队里取出该点。等权时 dist 已最优。 |
| `if current == goal` | 第一次取出终点即可返回。 |
| `for nxt, _ in neighbors` | 看所有出边。`_cost` 本课不用，留给 Dijkstra。 |
| `if nxt in came_from: continue` | 见过就丢。第一次见才是最短。 |
| 写入 came_from / dist 再 append | **先标记再入队**，防止同一点进队两次。 |

`deque` 来自 `collections`。列表 `pop(0)` 是 \(O(n)\)，不要用。


In [ ]:
def bfs(start, goal, verbose=True):
    queue = deque([start])
    came_from = {start: None}
    dist = {start: 0}
    step = 0
    while queue:
        current = queue.popleft()
        step += 1
        if verbose:
            print_step(
                step,
                f"扩展 {current}（dist={dist[current]}）",
                draw_grid(
                    W, H,
                    obstacles=OBSTACLES,
                    start=start,
                    goal=goal,
                    open_nodes=list(queue),
                    closed=list(came_from),
                    current=current,
                ),
                extra=f"queue={list(queue)}",
            )
        if current == goal:
            path = reconstruct(came_from, goal)
            if verbose:
                print("路径", path, "边数", dist[goal])
                print(draw_grid(W, H, obstacles=OBSTACLES, start=start, goal=goal, path=path))
            return path, dist[goal]
        for nxt, _cost in neighbors(current):
            if nxt in came_from:
                continue
            came_from[nxt] = current
            dist[nxt] = dist[current] + 1
            queue.append(nxt)
    if verbose:
        print("失败：队列空，终点从未入队。")
    return None, None


path, cost = bfs(START, GOAL)


对照第 5 节表：第 1 步扩展 (0,0)，队列变成 (1,0),(0,1)。字符图里 `o` 是队列，`x` 是已发现（含已扩展），`*` 是当前。

---

## 9. 失败、复杂度、常见 bug

**不可达：** 中间一堵墙。队列处理完左半边就空了。


In [ ]:
def reachable(start, goal, walls, w=3, h=3):
    def ok(p):
        x, y = p
        return 0 <= x < w and 0 <= y < h and p not in walls
    q = deque([start])
    seen = {start}
    while q:
        c = q.popleft()
        if c == goal:
            return True
        x, y = c
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            n = (x + dx, y + dy)
            if ok(n) and n not in seen:
                seen.add(n)
                q.append(n)
    return False


print("竖墙隔开:", reachable((0, 1), (2, 1), {(1, 0), (1, 1), (1, 2)}))
print("去掉中间墙:", reachable((0, 1), (2, 1), {(1, 0), (1, 2)}))


**常见 bug**

1. 用栈 `list.pop()` 当队列 → 变成 DFS，边数不必最短。
2. 出队才写入 visited → 同一点多次入队，最坏指数级。
3. `GRID[x][y]` 写反。
4. 越界不检查，`GRID[-1]` 取到最后一行。
5. 边权不是常数还用 BFS。

**库实现对照：** `lib/search.py` 的 `bfs()` 与上面相同，多了 `frames` 给调度台。`graph.neighbors(u)` 就是本课的 `neighbors`。


In [ ]:
from lib.maps import grid_open
from lib.search import bfs as bfs_lib
from lib.studio.widget import PlannerStudio

g = grid_open(8, 8)
result = bfs_lib(g, (0, 0), (7, 7))
print("空 8x8 最短边数应为 14:", result.cost, "扩展", result.expanded)

studio = PlannerStudio()
studio.show_search({**g.to_studio(), "start": [0, 0], "goal": [7, 7]}, result, "BFS")
studio


## 10. 条目小结

| 项目 | 内容 |
|------|------|
| 保证 | 等权图上边数最优、有限图完备 |
| 不保证 | 不等权最短路 |
| 结构 | FIFO 队列 + 入队时标记 |
| 时间 | \(O(V+E)\) |
| 下一课 | 边权不同 → Dijkstra，队列换成堆 |

## 练习

1. 邻居改成上、下、左、右，边数变不变？路径点列变不变？
2. 八连通（含斜向，代价仍 1）从 (0,0) 到 (3,2) 最少几步？`neighbors` 要加哪四个偏移？
3. 证明：若某点以 dist=k 入队，则不存在边数 < k 的 s–该点路径。（对 k 归纳。）
